In [24]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.cmac import CMAC

In [25]:
BLOCK_SIZE = 16         # AES Block size in bytes
RB = 0x87               # CMAC reduction constant for 128 bit block ciphers

In [26]:
def aes_encrypt_block(key: bytes, block: bytes) -> bytes:
    cipher = Cipher(algorithms.AES(key), modes.ECB())
    encryptor = cipher.encryptor()
    return encryptor.update(block) + encryptor.finalize()

def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x,y in zip(a,b))

def left_shift_one_bit(block: bytes) -> bytes:
    value = int.from_bytes(block, "big")
    value = (value << 1) & ((1 << 128) - 1)
    return value.to_bytes(16, "big")

def generate_subkey(key: bytes):
    zero_block = b"\x00" * 16

    # L = AES_K(0^128)
    L = aes_encrypt_block(key, zero_block)

    # Generate K1
    if L[0] & 0x80:
        K1 = left_shift_one_bit(L)
        K1 = K1[:-1] +  bytes([K1[-1] ^RB])
    else:
        K1 = left_shift_one_bit(L)
    
    # Generate K2
    if K1[0] & 0x80:
        K2 = left_shift_one_bit(K1)
        K2 = K2[:-1] +  bytes([K2[-1] ^RB])
    else:
        K2 = left_shift_one_bit(K1)
    
    return L, K1, K2

def pad_cmac(block: bytes) -> bytes:
    # CMAC padding: append 0x80, then zeros
    return block + b"\x80" + b"\x00" * (BLOCK_SIZE - len(block) - 1)

def cmac_manual(key: bytes, message: bytes) -> bytes:
    L, K1, K2 = generate_subkey(key)

    print(f"L = {L.hex()}")
    print(f"K1 = {K1.hex()}")
    print(f"K2 = {K2.hex()}")
    print("\n")

    if(message) == 0:
        blocks = []
        last_block_complete = False
    else:
        blocks = [
            message[i:i + BLOCK_SIZE]
            for i in range(0, len(message), BLOCK_SIZE)
        ]
        last_block_complete = len(blocks[-1]) == BLOCK_SIZE

    if last_block_complete:
        last_block = xor_bytes(blocks[-1], K1)
        normal_block = blocks[:-1]

        print(f"M_LAST = {blocks[-1].hex()}")
        print(f"M_last XOR K1 = {last_block.hex()}")
        print("\n")
        
    else:
        partial_last = blocks[-1] if blocks else b""
        padded = pad_cmac(partial_last)
        last_block = xor_bytes(padded, K2)
        normal_blocks = blocks[:-1] if blocks else []

        print(f"Partial Last Block = {partial_last.hex()}")
        print(f"Padded Block = {padded.hex()}")
        print(f"Last Block = {last_block.hex()}")
        print("\n")

    X = b"\x00" * BLOCK_SIZE

    print("=== CBC-Like AES Chaining===")
    print(f"X0 = {X.hex()}")

    for i, block in enumerate(normal_blocks, start=1):
        y = xor_bytes(X, block)
        X = aes_encrypt_block(key, y)

        print(f"Block M{i} = {block.hex()}")
        print(f"X{i-1} XOR M{i} = {y.hex()}")
        print(f"X{i} = AES_K(...) = {X.hex()}")
        print("\n")

    y = xor_bytes(X, last_block)
    tag = aes_encrypt_block(key, y)

    print("=== Final ===")
    print(f"X xor Final Block = {y.hex()}")
    print(f"TAG = AES_K(....) = {tag.hex()}")
    print("\n")

    return tag

def cmac_libray(key: bytes, messages: bytes) -> bytes:
    c = CMAC(algorithms.AES(key))
    c.update(messages)
    return c.finalize()

In [37]:
KEY = bytes.fromhex("2b7e151628aed2a6abf7158809cf4f3c")

message = b"Hello AES-CMAC1"

manual_tag = cmac_manual(KEY, message)
library_tag = cmac_libray(KEY, message)

print(manual_tag.hex)
print("====================================")
print(library_tag.hex())

if(manual_tag == library_tag):
    print("Tag Matched")
else:
    print("Different")


L = 7df76b0c1ab899b33e42f047b91b546f
K1 = fbeed618357133667c85e08f7236a8de
K2 = f7ddac306ae266ccf90bc11ee46d513b


Partial Last Block = 48656c6c6f204145532d434d414331
Padded Block = 48656c6c6f204145532d434d41433180
Last Block = bfb8c05c05c22789aa268253a52e60bb


=== CBC-Like AES Chaining===
X0 = 00000000000000000000000000000000
=== Final ===
X xor Final Block = bfb8c05c05c22789aa268253a52e60bb
TAG = AES_K(....) = bea2ea672764fdae246299993afa383d


<built-in method hex of bytes object at 0x0000022BCB54A630>
bea2ea672764fdae246299993afa383d
Tag Matched
